In [32]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [33]:
import pandas as pd

DATA_PATH = "../data/raw/api_error_logs_with_root_causes_220k_rows.csv"

df = pd.read_csv(
    DATA_PATH,
    low_memory=False
)

print(df.shape)

df.head()

(220000, 22)


,timestamp,api_name,service_owner,environment,http_method,endpoint,status_code,error_type,root_cause,latency_ms,...,retry_count,is_retry_successful,client_ip,region,container_id,host_id,thread_id,log_level,error_message,resolution_action
0,2024-01-01 00:00:00,inventory-api,team-beta,dev,DELETE,/v1/lljugd,503,Timeout,High latency in network,9403,...,1,True,192.168.97.252,ap-south-1,xuqbiprmtjwu,zjcdzcnlzt,1258,WARN,Internal server error,Refresh token
1,2024-01-01 00:00:05,user-api,team-alpha,staging,DELETE,/v1/mckssy,500,ClientError,Database connection failure,12798,...,2,False,192.168.134.37,us-west-2,tkpvinlsimyz,gcpwfmdubo,8020,INFO,Internal server error,Fix configuration
2,2024-01-01 00:00:10,order-api,team-alpha,prod,DELETE,/v1/ejvxxt,401,AuthError,Schema validation error,1746,...,4,False,192.168.199.184,us-east-1,tocjxcugfqfx,gqalvwpoum,5427,ERROR,Unauthorized access,Restart service
3,2024-01-01 00:00:15,user-api,team-beta,prod,GET,/v1/noqarz,400,RateLimit,Database connection failure,8021,...,0,True,192.168.206.244,eu-central-1,lbgdfttkzuto,namadinzvh,8447,WARN,Bad gateway,Optimize query
4,2024-01-01 00:00:20,user-api,team-gamma,prod,GET,/v1/rcqwsq,401,ClientError,Missing authorization scope,4924,...,0,False,192.168.63.219,us-west-2,oepkhhmcyzlr,nuqorgcfko,8747,WARN,Internal server error,Restart service


In [34]:
safe_features = [
    "api_name",
    "service_owner",
    "environment",
    "http_method",
    "endpoint",
    "latency_ms",
    "request_size_bytes",
    "response_size_bytes",
    "retry_count",
    "region",
    "log_level"
]

target_column = "root_cause"

In [35]:
X = df[safe_features]

y = df[target_column]

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [37]:
categorical_features = [
    "api_name",
    "service_owner",
    "environment",
    "http_method",
    "endpoint",
    "region",
    "log_level"
]

numerical_features = [
    "latency_ms",
    "request_size_bytes",
    "response_size_bytes",
    "retry_count"
]

In [38]:
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [39]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [40]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [41]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [42]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['latency_ms',
                                                   'request_size_bytes',
                                                   'response_size_bytes',
                                                   'retry_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['api_name', 'service_owner',
                                                   'environment', 'http_method',
                                                   'endpoint', 'region',
                                                   'log_level'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [43]:
y_pred = model_pipeline.predict(X_test)

In [44]:
print(classification_report(y_test, y_pred))

                                     precision    recall  f1-score   support

                     CPU throttling       0.08      0.06      0.06      2918
             DNS resolution failure       0.06      0.08      0.07      2937
        Database connection failure       0.06      0.05      0.06      2924
                  Deadlock detected       0.06      0.02      0.03      2910
     Downstream service unavailable       0.06      0.08      0.07      2909
       Expired authentication token       0.06      0.08      0.07      2972
            High latency in network       0.07      0.07      0.07      2951
            Invalid request payload       0.07      0.06      0.06      2913
                  Memory exhaustion       0.07      0.10      0.08      2972
Misconfigured environment variables       0.07      0.04      0.05      2918
        Missing authorization scope       0.07      0.04      0.05      2909
             Null pointer exception       0.07      0.14      0.09      295

In [45]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.0669090909090909


## Observation

The baseline Logistic Regression model using only infrastructure metadata performed near random-chance accuracy (~6–7%).

This suggests that infrastructure metadata alone does not contain sufficient predictive signal for reliable root cause classification.

To improve performance, operational context features such as HTTP status codes and error types will be introduced in the next experiment.

# Experiment 2 — Operational Context Features

In this experiment, operational telemetry signals such as HTTP status codes and error types are introduced alongside infrastructure metadata.

These features are expected to contain stronger predictive signals related to API failures and system behavior, potentially improving root cause classification performance.

In [46]:
operational_features = [
    "api_name",
    "service_owner",
    "environment",
    "http_method",
    "endpoint",
    "status_code",
    "error_type",
    "latency_ms",
    "request_size_bytes",
    "response_size_bytes",
    "retry_count",
    "region",
    "log_level"
]

target_column = "root_cause"

In [47]:
X = df[operational_features]

y = df[target_column]

In [48]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [49]:
categorical_features = [
    "api_name",
    "service_owner",
    "environment",
    "http_method",
    "endpoint",
    "error_type",
    "region",
    "log_level"
]

In [50]:
numerical_features = [
    "status_code",
    "latency_ms",
    "request_size_bytes",
    "response_size_bytes",
    "retry_count"
]

In [51]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [52]:
model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

In [53]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['status_code', 'latency_ms',
                                                   'request_size_bytes',
                                                   'response_size_bytes',
                                                   'retry_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['api_name', 'service_owner',
                                                   'environment', 'http_method',
                                                   'endpoint', 'error_type',
                                                   'region', 'log_level'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [54]:
print(classification_report(y_test, y_pred))

                                     precision    recall  f1-score   support

                     CPU throttling       0.08      0.06      0.06      2918
             DNS resolution failure       0.06      0.08      0.07      2937
        Database connection failure       0.06      0.05      0.06      2924
                  Deadlock detected       0.06      0.02      0.03      2910
     Downstream service unavailable       0.06      0.08      0.07      2909
       Expired authentication token       0.06      0.08      0.07      2972
            High latency in network       0.07      0.07      0.07      2951
            Invalid request payload       0.07      0.06      0.06      2913
                  Memory exhaustion       0.07      0.10      0.08      2972
Misconfigured environment variables       0.07      0.04      0.05      2918
        Missing authorization scope       0.07      0.04      0.05      2909
             Null pointer exception       0.07      0.14      0.09      295

In [55]:
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.0669090909090909


In [56]:
print(X.columns.tolist())

['api_name', 'service_owner', 'environment', 'http_method', 'endpoint', 'status_code', 'error_type', 'latency_ms', 'request_size_bytes', 'response_size_bytes', 'retry_count', 'region', 'log_level']


In [57]:
print(X.shape)

(220000, 13)


In [58]:
print(X.head())

        api_name service_owner environment http_method    endpoint  \
0  inventory-api     team-beta         dev      DELETE  /v1/lljugd   
1       user-api    team-alpha     staging      DELETE  /v1/mckssy   
2      order-api    team-alpha        prod      DELETE  /v1/ejvxxt   
3       user-api     team-beta        prod         GET  /v1/noqarz   
4       user-api    team-gamma        prod         GET  /v1/rcqwsq   

   status_code   error_type  latency_ms  request_size_bytes  \
0          503      Timeout        9403               19735   
1          500  ClientError       12798                 898   
2          401    AuthError        1746               42249   
3          400    RateLimit        8021               34209   
4          401  ClientError        4924               11789   

   response_size_bytes  retry_count        region log_level  
0                56072            1    ap-south-1      WARN  
1                47461            2     us-west-2      INFO  
2             

# Experiment 3 — Random Forest Classifier

Linear models showed limited predictive performance even after introducing operational context features.

To capture nonlinear relationships and feature interactions, a Random Forest classifier is introduced as the next baseline model.

In [59]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [60]:
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['status_code', 'latency_ms',
                                                   'request_size_bytes',
                                                   'response_size_bytes',
                                                   'retry_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['api_name', 'service_owner',
                                                   'environment', 'http_method',
                                                   'endpoint', 'error_type',
                                                   'region', 'log_level'])])),
                ('classifier',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

In [61]:
rf_y_pred = rf_pipeline.predict(X_test)

In [62]:
print(classification_report(y_test, rf_y_pred))

                                     precision    recall  f1-score   support

                     CPU throttling       0.07      0.08      0.07      2918
             DNS resolution failure       0.07      0.09      0.08      2937
        Database connection failure       0.06      0.07      0.06      2924
                  Deadlock detected       0.07      0.07      0.07      2910
     Downstream service unavailable       0.07      0.07      0.07      2909
       Expired authentication token       0.07      0.08      0.08      2972
            High latency in network       0.06      0.07      0.07      2951
            Invalid request payload       0.07      0.06      0.06      2913
                  Memory exhaustion       0.07      0.07      0.07      2972
Misconfigured environment variables       0.06      0.05      0.06      2918
        Missing authorization scope       0.07      0.06      0.06      2909
             Null pointer exception       0.06      0.06      0.06      295

In [63]:
print("Random Forest Accuracy:", accuracy_score(y_test, rf_y_pred))

Random Forest Accuracy: 0.06563636363636363


# Experiment 4 — Leakage-Prone Features

This experiment intentionally introduces post-incident diagnostic fields such as error messages, retry outcomes, and resolution actions.

These features are expected to significantly inflate performance because they may contain direct or indirect information about the target root cause.

The purpose of this experiment is to demonstrate how data leakage can create misleadingly high evaluation metrics in machine learning systems.

In [81]:
leakage_features = [
    "api_name",
    "service_owner",
    "environment",
    "http_method",
    "endpoint",
    "status_code",
    "error_type",
    "latency_ms",
    "request_size_bytes",
    "response_size_bytes",
    "retry_count",
    "is_retry_successful",
    "region",
    "log_level",
    "error_message",
    "resolution_action"
]

target_column = "root_cause"

X = df[leakage_features]

y = df[target_column]

In [73]:
print(X.columns.tolist())

['api_name', 'service_owner', 'environment', 'http_method', 'endpoint', 'status_code', 'error_type', 'latency_ms', 'request_size_bytes', 'response_size_bytes', 'retry_count', 'is_retry_successful', 'region', 'log_level', 'error_message', 'resolution_action']


In [82]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [83]:
categorical_features = [
    "api_name",
    "service_owner",
    "environment",
    "http_method",
    "endpoint",
    "error_type",
    "region",
    "log_level",
    "error_message",
    "resolution_action"
]

In [84]:
numerical_features = [
    "status_code",
    "latency_ms",
    "request_size_bytes",
    "response_size_bytes",
    "retry_count"
]

In [85]:
boolean_features = [
    "is_retry_successful"
]

In [86]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
        ("bool", "passthrough", boolean_features)
    ]
)

In [87]:
rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [88]:
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['status_code', 'latency_ms',
                                                   'request_size_bytes',
                                                   'response_size_bytes',
                                                   'retry_count']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['api_name', 'service_owner',
                                                   'environment', 'http_method',
                                                   'endpoint', 'error_type',
                                                   'region', 'log_level',
                                                   'error_message',
                                                   'resolution_action']),
                                                 ('bool', 'passthrough',
                                                  ['is_retry_successful'])])),
                ('classifier',
                 RandomForestClassifier(n_jobs=-1, random_state=42))])

In [89]:
rf_y_pred = rf_pipeline.predict(X_test)

In [90]:
print(classification_report(y_test, rf_y_pred))

                                     precision    recall  f1-score   support

                     CPU throttling       0.07      0.08      0.08      2918
             DNS resolution failure       0.07      0.08      0.07      2937
        Database connection failure       0.06      0.07      0.07      2924
                  Deadlock detected       0.07      0.07      0.07      2910
     Downstream service unavailable       0.07      0.07      0.07      2909
       Expired authentication token       0.07      0.08      0.07      2972
            High latency in network       0.07      0.08      0.07      2951
            Invalid request payload       0.07      0.06      0.07      2913
                  Memory exhaustion       0.07      0.08      0.07      2972
Misconfigured environment variables       0.06      0.05      0.06      2918
        Missing authorization scope       0.07      0.06      0.06      2909
             Null pointer exception       0.07      0.06      0.06      295

In [91]:
print("Random Forest Accuracy:", accuracy_score(y_test, rf_y_pred))

Random Forest Accuracy: 0.0676590909090909


# Experiment 5 — TF-IDF Log Intelligence Pipeline

Previous experiments showed that structured infrastructure metadata contained very limited predictive signal for root cause classification.

In this experiment, the `error_message` field is treated as natural language text using TF-IDF vectorization to capture semantic patterns and operational terminology from API logs.

In [92]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [93]:
text_features = [
    "error_message"
]

structured_features = [
    "status_code",
    "latency_ms",
    "retry_count"
]

target_column = "root_cause"

In [94]:
X = df[text_features + structured_features]

y = df[target_column]

In [95]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [96]:
text_transformer = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                stop_words="english",
                max_features=500
            )
        )
    ]
)

In [97]:
numerical_features = [
    "status_code",
    "latency_ms",
    "retry_count"
]

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [98]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "text",
            text_transformer,
            "error_message"
        ),
        (
            "num",
            numerical_transformer,
            numerical_features
        )
    ]
)

In [99]:
nlp_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000
            )
        )
    ]
)

In [100]:
nlp_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('text',
                                                  Pipeline(steps=[('tfidf',
                                                                   TfidfVectorizer(max_features=500,
                                                                                   stop_words='english'))]),
                                                  'error_message'),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['status_code', 'latency_ms',
                                                   'retry_count'])])),
                ('classifier', LogisticRegression(max_iter=1000))])

In [101]:
nlp_y_pred = nlp_pipeline.predict(X_test)

In [102]:
print(classification_report(y_test, nlp_y_pred))

                                     precision    recall  f1-score   support

                     CPU throttling       0.07      0.03      0.04      2918
             DNS resolution failure       0.08      0.02      0.03      2937
        Database connection failure       0.05      0.00      0.00      2924
                  Deadlock detected       0.07      0.09      0.08      2910
     Downstream service unavailable       0.07      0.06      0.07      2909
       Expired authentication token       0.07      0.18      0.10      2972
            High latency in network       0.07      0.14      0.09      2951
            Invalid request payload       0.08      0.04      0.06      2913
                  Memory exhaustion       0.06      0.08      0.07      2972
Misconfigured environment variables       0.06      0.04      0.05      2918
        Missing authorization scope       0.06      0.01      0.01      2909
             Null pointer exception       0.07      0.09      0.08      295

In [103]:
print("TF-IDF Pipeline Accuracy:", accuracy_score(y_test, nlp_y_pred))

TF-IDF Pipeline Accuracy: 0.06863636363636363


# Final Observations and Conclusions

This project explored the feasibility of predicting API failure root causes using synthetic cloud infrastructure telemetry and log data.

Multiple machine learning approaches were evaluated, including:

- Logistic Regression
- Random Forest
- TF-IDF based NLP pipelines

Several feature subsets were tested:
- infrastructure metadata
- operational telemetry
- post-incident diagnostic fields
- textual log intelligence

Despite progressively introducing richer features and more advanced preprocessing techniques, all experiments consistently performed near random-chance accuracy (~6–7%). This strongly suggests that the dataset contains very limited predictive signal or weak causal relationships between the provided features and the target root causes.

From an engineering and analytical perspective, this project highlights several important machine learning concepts:

- the importance of validating dataset realism
- why feature quality matters more than model complexity
- how to investigate data leakage systematically
- how to structure reproducible ML pipelines
- the limitations of synthetic observability datasets

Rather than artificially inflating results or overfitting models, the project focused on rigorous experimentation, controlled feature analysis, and honest evaluation of model behavior.

Future improvements could include:
- using real production observability datasets
- sequence-aware log modeling
- transformer-based log embeddings
- temporal incident analysis
- retrieval-augmented incident diagnosis systems
- anomaly detection workflows for cloud telemetry
